# Лабораторная работа №7: Автоматическая разметка изображений с помощью VLM

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.

## Цели обучения

После работы студент должен уметь:

- Определить schema аннотации.
- Получить структурированные ответы VLM с retry.
- Проверить выборку человеком и обучить downstream baseline.


## Теоретическая часть

Авторазметка — это data pipeline: schema → generation → parsing → validation → confidence proxy → human audit. Нельзя терять сырой ответ модели: он нужен для трассировки ошибок и повторной обработки.


### Математическая постановка

Agreement для поля можно оценивать Cohen $\kappa=(p_o-p_e)/(1-p_e)$. Для редких классов важнее class-wise precision/recall, чем общая accuracy.


### Материалы

- [JSON Schema](https://json-schema.org/learn/getting-started-step-by-step)
- [Pydantic](https://docs.pydantic.dev/latest/)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практический протокол

Полный эксперимент (`FAST_DEV_RUN=False`) требует GPU и ручной экспертизы captions. Классы CIFAR-10 служат проверяемым эталоном, а не ответами VLM. Установите torchvision, transformers, accelerate, bitsandbytes, pydantic, scikit-learn. В Colab сохраните `artifacts/` на Drive до начала инференса. FAST_DEV_RUN проверяет загрузку данных и схему без фиктивных метрик.

In [1]:
from pathlib import Path
import json, os, random, time
import numpy as np
SEED = 42
FAST_DEV_RUN = False  # True: проверка механики на малой выборке, НЕ отчётный эксперимент
ARTIFACTS = Path('artifacts'); ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
def save_json(name, data):
    (ARTIFACTS/name).write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str))

VLM_LOADER = "\n# Одна и та же квантованная модель и интерфейс используются для сравниваемых запусков.\n# В Colab предварительно: %pip install -q 'transformers>=4.51,<5' accelerate bitsandbytes pillow datasets\nimport torch\nfrom PIL import Image\nfrom transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig\nMODEL_ID = os.getenv('CV_VLM_MODEL', 'Qwen/Qwen2.5-VL-3B-Instruct')\nassert torch.cuda.is_available(), 'Требуется GPU для реального VLM-инференса; CPU smoke проверяет только данные и код.'\nquant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)\nprocessor = AutoProcessor.from_pretrained(MODEL_ID)\nmodel = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, quantization_config=quant, device_map='auto').eval()\ndef infer_image(path, prompt, adapter=None):\n    messages = [{'role':'user','content':[{'type':'image','image':Image.open(path).convert('RGB')},{'type':'text','text':prompt}]}]\n    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n    inputs = processor(text=[text], images=[Image.open(path).convert('RGB')], return_tensors='pt').to(model.device)\n    with torch.no_grad(): generated = model.generate(**inputs, max_new_tokens=96, do_sample=False)\n    return processor.batch_decode(generated[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()\n"

## Оценивание

Десять обязательных предметных этапов — по 1 баллу каждый, всего 10 баллов. Отдельное творческое исследование — до 1 дополнительного балла. Каждый этап принимается по указанному наблюдаемому результату.

### Источник и классы

*Вес: 1 балл.*

**Постановка.** Этап «источник и классы» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Загрузите раздельные части CIFAR-10 и зафиксируйте перечень классов.
2. Сохраните и предъявите train/test CIFAR-10 и список классов.

**Результат.** train/test CIFAR-10 и список классов.

**Критерий принятия.** Задание считается принятым, если train/test CIFAR-10 и список классов получены из фактических входов этого этапа и доступны для проверки.

In [2]:
import torchvision
DATA_ROOT = Path(os.getenv('CV_COURSE_DATA', str(Path.cwd().parents[1] / 'data')))
raw_train = torchvision.datasets.CIFAR10(DATA_ROOT, train=True, download=True)
raw_test = torchvision.datasets.CIFAR10(DATA_ROOT, train=False, download=True)
CLASSES = list(raw_train.classes)
save_json("classes.json", {"classes": CLASSES, "source": "CIFAR-10 (Krizhevsky, 2009), torchvision", "train_size": len(raw_train), "test_size": len(raw_test)})
print("классы:", CLASSES, "| train:", len(raw_train), "test:", len(raw_test))

классы: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck'] | train: 50000 test: 10000


### Стратифицированный отбор изображений

*Вес: 1 балл.*

**Постановка.** Этап «стратифицированный отбор изображений» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Отберите равное число изображений каждого класса в train и test; запишите постоянные ID и пути.
2. Сохраните и предъявите пути и идентификаторы изображений по классам.

**Результат.** пути и идентификаторы изображений по классам.

**Критерий принятия.** Задание считается принятым, если пути и идентификаторы изображений по классам получены из фактических входов этого этапа и доступны для проверки.

In [3]:
PER_CLASS = {"train": 4, "test": 2} if FAST_DEV_RUN else {"train": 40, "test": 20}
rng = np.random.RandomState(SEED)
selection = {}
for split, ds in (("train", raw_train), ("test", raw_test)):
    targets = np.array(ds.targets)
    selection[split] = {c: sorted(rng.choice(np.where(targets == k)[0], PER_CLASS[split], replace=False).tolist()) for k, c in enumerate(CLASSES)}
IMG_DIR = DATA_ROOT / "lab07" / "images"
IMG_DIR.mkdir(parents=True, exist_ok=True)
VLM_SIZE = 224   # 32×32 слишком мало для патчей 14 px: увеличиваю бикубически до 224 (8×8 токенов после merge)
from PIL import Image
paths = {}
for split, ds in (("train", raw_train), ("test", raw_test)):
    for c, idxs in selection[split].items():
        for i in idxs:
            uid = f"cifar10-{split}-{i:05d}"
            p = IMG_DIR / f"{uid}.png"
            if not p.exists():
                Image.fromarray(ds.data[i]).resize((VLM_SIZE, VLM_SIZE), Image.BICUBIC).save(p)
            paths[uid] = str(p)
print({s: {c: len(v) for c, v in sel.items()} for s, sel in selection.items()})
print("пример:", next(iter(paths.items())))

{'train': {'airplane': 40, 'automobile': 40, 'bird': 40, 'cat': 40, 'deer': 40, 'dog': 40, 'frog': 40, 'horse': 40, 'ship': 40, 'truck': 40}, 'test': {'airplane': 20, 'automobile': 20, 'bird': 20, 'cat': 20, 'deer': 20, 'dog': 20, 'frog': 20, 'horse': 20, 'ship': 20, 'truck': 20}}
пример: ('cifar10-train-01097', '/Users/filipp/Desktop/рабочий стол/Вуз/учеба/computer_vision/computer_vision_2026/data/lab07/images/cifar10-train-01097.png')


### Манифест изображений

*Вес: 1 балл.*

**Постановка.** Этап «манифест изображений» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Проверьте объём выборок и сохраните манифест с эталонными метками, не передавая их VLM.
2. Сохраните и предъявите artifacts/images.jsonl без пересечения train/test.

**Результат.** artifacts/images.jsonl без пересечения train/test.

**Критерий принятия.** Задание считается принятым, если artifacts/images.jsonl без пересечения train/test получены из фактических входов этого этапа и доступны для проверки.

In [4]:
records = []
for split, ds in (("train", raw_train), ("test", raw_test)):
    for c, idxs in selection[split].items():
        for i in idxs:
            uid = f"cifar10-{split}-{i:05d}"
            records.append({"id": uid, "split": split, "cifar_index": int(i), "path": paths[uid], "reference_class": c,
                            "source": "CIFAR-10 " + split, "vlm_input_size": VLM_SIZE})
ids = {s: {r["id"] for r in records if r["split"] == s} for s in ("train", "test")}
assert not ids["train"] & ids["test"]
assert all(Path(r["path"]).is_file() for r in records)
with open(ARTIFACTS / "images.jsonl", "w") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
# эталонная метка хранится в манифесте, но в запрос к VLM не передаётся
print(f"манифест: {len(records)} изображений, train {len(ids['train'])}, test {len(ids['test'])}, пересечений нет")

манифест: 600 изображений, train 400, test 200, пересечений нет


### Схема и парсер VLM-ответа

*Вес: 1 балл.*

**Постановка.** Этап «схема и парсер vlm-ответа» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Определите строгую схему class_name, caption, uncertain; отклоняйте незнакомые классы и пустые описания.
2. Сохраните и предъявите типизированная схема, prompt и функция parse.

**Результат.** типизированная схема, prompt и функция parse.

**Критерий принятия.** Задание считается принятым, если типизированная схема, prompt и функция parse получены из фактических входов этого этапа и доступны для проверки.

In [5]:
from typing import Literal
from pydantic import BaseModel, Field, ValidationError, field_validator

class Annotation(BaseModel):
    class_name: Literal[tuple(CLASSES)]
    caption: str = Field(min_length=3, max_length=300)
    uncertain: bool
    @field_validator("caption")
    @classmethod
    def not_blank(cls, v):
        if not v.strip():
            raise ValueError("пустое описание")
        return v.strip()

SCHEMA = Annotation.model_json_schema()
PROMPT = ("Classify this small photo into exactly one of these classes: " + ", ".join(CLASSES) + ".\n"
          "Return only a JSON object with keys: \"class_name\" (one of the classes above, lowercase), "
          "\"caption\" (one short sentence describing what is visible) and \"uncertain\" (true if you are not sure about the class).")

def parse(raw):
    """Достаёт первый JSON-объект из ответа и валидирует его схемой; возвращает (Annotation | None, ошибка)."""
    import re
    text = raw.strip()
    text = re.sub(r"^```(?:json)?|```$", "", text, flags=re.M).strip()
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        return None, "нет JSON-объекта"
    try:
        obj = json.loads(m.group(0))
    except json.JSONDecodeError as e:
        return None, f"невалидный JSON: {e.msg}"
    if isinstance(obj.get("class_name"), str):
        obj["class_name"] = obj["class_name"].strip().lower()
    try:
        return Annotation(**obj), None
    except ValidationError as e:
        return None, "схема: " + "; ".join(err["msg"] for err in e.errors())

save_json("schema.json", {"json_schema": SCHEMA, "prompt": PROMPT})
print(json.dumps(SCHEMA, indent=1)[:600])

{
 "properties": {
  "class_name": {
   "enum": [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck"
   ],
   "title": "Class Name",
   "type": "string"
  },
  "caption": {
   "maxLength": 300,
   "minLength": 3,
   "title": "Caption",
   "type": "string"
  },
  "uncertain": {
   "title": "Uncertain",
   "type": "boolean"
  }
 },
 "required": [
  "class_name",
  "caption",
  "uncertain"
 ],
 "title": "Annotation",
 "type": "object"
}


### Возобновляемое получение аннотаций

*Вес: 1 балл.*

**Постановка.** Этап «возобновляемое получение аннотаций» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Вызовите Qwen2.5-VL для каждого изображения, повторяйте ошибочный JSON до трёх раз и сохраняйте исходные ответы по ID.
2. Сохраните и предъявите artifacts/annotations.jsonl с сырыми ответами и временем.

**Результат.** artifacts/annotations.jsonl с сырыми ответами и временем.

**Критерий принятия.** Задание считается принятым, если artifacts/annotations.jsonl с сырыми ответами и временем получены из фактических входов этого этапа и доступны для проверки.

In [6]:
import torch, transformers
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
# Тот же загрузчик, что в VLM_LOADER шаблона, но с выбором устройства CUDA → MPS (bitsandbytes ≥ 0.50 умеет NF4 на MPS)
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
assert DEVICE != "cpu", "Требуется GPU для реального VLM-инференса; CPU smoke проверяет только данные и код."
MODEL_ID = os.getenv("CV_VLM_MODEL", "Qwen/Qwen2.5-VL-3B-Instruct")
# NF4 (bitsandbytes) на MPS работает, но веса деквантуются на каждом шаге декодирования: в пробном прогоне разметка
# занимала около 2 минут на изображение. Для 600 изображений на Apple GPU беру ту же модель в fp16 (48 ГБ памяти это
# позволяют); на CUDA остаётся NF4, как в загрузчике шаблона.
USE_NF4 = DEVICE == "cuda"
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16) if USE_NF4 else None
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, quantization_config=quant, dtype=torch.float16, device_map={"": DEVICE}).eval()
FIRST_TOKEN = {c: processor.tokenizer.encode(c, add_special_tokens=False)[0] for c in CLASSES}
assert len(set(FIRST_TOKEN.values())) == len(CLASSES), "первые токены названий классов должны различаться"

import re

@torch.no_grad()
def annotate_once(path, prompt, sample=False):
    image = Image.open(path).convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(DEVICE)
    gen = dict(max_new_tokens=64, do_sample=sample, output_scores=True, return_dict_in_generate=True)
    if sample:
        gen.update(temperature=0.7, top_p=0.9)
    out = model.generate(**inputs, **gen)
    new = out.sequences[0, inputs.input_ids.shape[1]:]
    raw = processor.decode(new, skip_special_tokens=True).strip()
    # confidence proxy: в позиции, где начинается значение class_name, сравниваю вероятности первых токенов
    # десяти названий классов; conf — доля массы выбранного класса среди них
    conf, class_probs = None, None
    for k in range(len(new)):
        if re.search(r'"class_name"\s*:\s*"$', processor.decode(new[:k], skip_special_tokens=True)):
            p = torch.softmax(out.scores[k][0].float(), -1)
            mass = np.array([p[FIRST_TOKEN[c]].item() for c in CLASSES])
            class_probs = (mass / mass.sum()).round(5).tolist()
            conf = float(max(class_probs))
            break
    return raw, conf, class_probs

def annotate(rec, max_attempts=3):
    attempts = []
    for a in range(max_attempts):
        t0 = time.time()
        raw, conf, class_probs = annotate_once(rec["path"], PROMPT, sample=a > 0)   # повтор с семплированием, иначе ответ не изменится
        ann, err = parse(raw)
        attempts.append({"attempt": a + 1, "raw": raw, "error": err, "seconds": time.time() - t0, "class_conf": conf, "class_probs": class_probs})
        if ann is not None:
            return {"id": rec["id"], "split": rec["split"], "valid": True, "annotation": ann.model_dump(), "class_conf": conf, "attempts": attempts}
    return {"id": rec["id"], "split": rec["split"], "valid": False, "annotation": None, "class_conf": None, "attempts": attempts}

ann_path = ARTIFACTS / "annotations.jsonl"
done = {json.loads(l)["id"] for l in ann_path.read_text().splitlines()} if ann_path.exists() else set()
t_start = time.time()
with open(ann_path, "a") as f:
    for rec in records:
        if rec["id"] in done:
            continue
        res = annotate(rec)
        res["model"] = MODEL_ID; res["seconds_total"] = sum(a["seconds"] for a in res["attempts"])
        f.write(json.dumps(res, ensure_ascii=False) + "\n"); f.flush()
annotations = {json.loads(l)["id"]: json.loads(l) for l in ann_path.read_text().splitlines()}
n_retry = sum(len(a["attempts"]) > 1 for a in annotations.values())
print(f"аннотаций: {len(annotations)}, валидных: {sum(a['valid'] for a in annotations.values())}, с повтором: {n_retry}, "
      f"время этого запуска {time.time() - t_start:.0f} c; модель {MODEL_ID}, transformers {transformers.__version__}")
save_json("annotation_run.json", {"model": MODEL_ID, "quantization": "bnb nf4" if USE_NF4 else "fp16 (MPS, без квантования)", "device": DEVICE, "prompt": PROMPT,
                                  "max_attempts": 3, "decoding": "attempt 1 greedy, retries: temperature 0.7, top_p 0.9", "max_new_tokens": 64,
                                  "transformers": transformers.__version__, "torch": torch.__version__,
                                  "mean_seconds_per_image": float(np.mean([a["seconds_total"] for a in annotations.values()]))})

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

аннотаций: 600, валидных: 600, с повтором: 3, время этого запуска 0 c; модель Qwen/Qwen2.5-VL-3B-Instruct, transformers 4.57.6


### Проверка формата без GPU

*Вес: 1 балл.*

**Постановка.** Этап «проверка формата без gpu» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. В smoke-режиме проверьте только разбор валидного ответа, не вычисляя качество VLM.
2. Сохраните и предъявите проверка JSON-схемы в smoke-режиме.

**Результат.** проверка JSON-схемы в smoke-режиме.

**Критерий принятия.** Задание считается принятым, если проверка JSON-схемы в smoke-режиме получены из фактических входов этого этапа и доступны для проверки.

In [7]:
samples = ['{"class_name": "cat", "caption": "A cat sitting on a sofa.", "uncertain": false}',
           '```json\n{"class_name": "Truck", "caption": "A red truck on a road.", "uncertain": true}\n```',
           '{"class_name": "tiger", "caption": "A tiger.", "uncertain": false}',
           '{"class_name": "dog", "caption": "   ", "uncertain": false}',
           'I think it is a ship.']
results = [parse(s) for s in samples]
for s, (ann, err) in zip(samples, results):
    print("OK " if ann else "REJECT", ann.model_dump() if ann else err)
assert results[0][0] is not None and results[1][0] is not None
assert all(r[0] is None for r in results[2:]), "неизвестный класс, пустое описание и текст без JSON должны отклоняться"
print("Проверка схемы пройдена; качество VLM здесь не вычисляется.")

OK  {'class_name': 'cat', 'caption': 'A cat sitting on a sofa.', 'uncertain': False}
OK  {'class_name': 'truck', 'caption': 'A red truck on a road.', 'uncertain': True}
REJECT схема: Input should be 'airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship' or 'truck'
REJECT схема: Value error, пустое описание
REJECT нет JSON-объекта
Проверка схемы пройдена; качество VLM здесь не вычисляется.


### Покрытие и матрица ошибок

*Вес: 1 балл.*

**Постановка.** Этап «покрытие и матрица ошибок» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Подсчитайте coverage, precision/recall и матрицу ошибок только по настоящим ответам модели на test.
2. Сохраните и предъявите artifacts/quality.json по ответам VLM.

**Результат.** artifacts/quality.json по ответам VLM.

**Критерий принятия.** Задание считается принятым, если artifacts/quality.json по ответам VLM получены из фактических входов этого этапа и доступны для проверки.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: artifacts/quality.json по ответам VLM
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Экспертная оценка описаний

*Вес: 1 балл.*

**Постановка.** Этап «экспертная оценка описаний» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Соберите отдельные экспертные оценки подписей и не подменяйте их пустым файлом.
2. Сохраните и предъявите artifacts/human_audit.jsonl с экспертными записями.

**Результат.** artifacts/human_audit.jsonl с экспертными записями.

**Критерий принятия.** Задание считается принятым, если artifacts/human_audit.jsonl с экспертными записями получены из фактических входов этого этапа и доступны для проверки.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: artifacts/human_audit.jsonl с экспертными записями
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Визуальные признаки для downstream

*Вес: 1 балл.*

**Постановка.** Этап «визуальные признаки для downstream» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Извлеките frozen-признаки ResNet18 для обучающих и отложенных изображений.
2. Сохраните и предъявите признаки ResNet18 train/test.

**Результат.** признаки ResNet18 train/test.

**Критерий принятия.** Задание считается принятым, если признаки ResNet18 train/test получены из фактических входов этого этапа и доступны для проверки.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: признаки ResNet18 train/test
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Классификаторы при равном бюджете

*Вес: 1 балл.*

**Постановка.** Этап «классификаторы при равном бюджете» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Обучите два классификатора на одних изображениях с ручными и VLM-метками; сравните на test.
2. Сохраните и предъявите artifacts/downstream.json и подтверждённый аудит.

**Результат.** artifacts/downstream.json и подтверждённый аудит.

**Критерий принятия.** Задание считается принятым, если artifacts/downstream.json и подтверждённый аудит получены из фактических входов этого этапа и доступны для проверки.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: artifacts/downstream.json и подтверждённый аудит
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Творческое мини-исследование

*Бонус: до 1 балла сверх 10.*

**Постановка.** Сформулируйте проверяемую гипотезу о предметном ограничении метода данной лабораторной и проверьте её на дополнительных реальных входах без использования отложенного теста для выбора гипотезы. Сопоставьте результаты с основным экспериментом и объясните расхождения.

**Результат.** Отдельный файл artifacts/bonus.md с гипотезой, протоколом, наблюдениями и выводом.

**Критерий принятия.** Задание считается принятым, если гипотеза проверена по сохранённым предсказаниям или изображениям и приведено воспроизводимое сравнение.

In [ ]:
# Реализуйте творческое мини-исследование по постановке выше.
raise NotImplementedError("Реализуйте бонусное исследование")

## Анализ результатов

Сопоставьте измеренные показатели на фиксированной отложенной выборке, укажите ограничения выборки и разберите ошибки по сохранённым предсказаниям. FAST_DEV_RUN не является основанием для итогового вывода.

## Требования к сдаче

Запустите полный режим, приложите конфигурацию, версии зависимостей, артефакты и инструкции повторного запуска. Не сдавайте фиктивные измерения.